In [1]:
import os
import json
import time
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patheffects as pe
import seaborn as sns
from rapidfuzz import fuzz, process
from sklearn.model_selection import cross_validate
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import SparsePCA
from sklearn.model_selection import train_test_split
from joblib import Parallel, delayed

from scholarlm.utils import (
    load_and_process_results,
    match_datasets,
    matching_precision_recall,
    get_filenames_in_directory,
    fit_temperature,
    apply_temperature,
    fit_temperature_from_probs,
    apply_temperature_from_probs,
)
from dotenv import load_dotenv
load_dotenv()

%load_ext autoreload
%autoreload 2

INFO 03-24 08:31:07 [__init__.py:220] No platform detected, vLLM is running on UnspecifiedPlatform
WARNING 03-24 08:31:21 [_custom_ops.py:20] Failed to import from vllm._C with ImportError('libcuda.so.1: cannot open shared object file: No such file or directory')


### Ground Truth Dataset

In [2]:
# ---------------------------------
# Load from ground truth dataset
# ---------------------------------

# Directory
with open(os.path.join("../data/nfix/directory.json"), "r") as f:
    paper_info = json.load(f)

def text_or_table_extraction(location):
    if 'figure' in location:
        return False
    if 'supplement' in location:
        return False
    if 'archive' in location:
        return False
    if 'author' in location:
        return False
    else:
        return True

registered_paper_info = {
    R: Rinfo for R,Rinfo in paper_info.items() if text_or_table_extraction(Rinfo['extraction_location']) 
}
registered_ids = list(registered_paper_info.keys())

#ground_truth_df = pd.read_csv("../data/nfix/nfix_data_corrected.csv", encoding_errors='ignore', index_col = 0)
#ground_truth_df = ground_truth_df.loc[ground_truth_df.title.isin(registered_titles)]
#ground_truth_df = ground_truth_df.reset_index(drop=True)

ground_truth_df = pd.read_csv("../data/nfix/meta/aquatic_N2fix_rates.csv")
ground_truth_df = ground_truth_df.loc[ground_truth_df.reference_id.isin(registered_ids)]

id_cols = [
        'nfix_rate_id', 'reference_id', 'site_name', 'latitude', 'longitude',
        'habitat', 'year', 'month', 'day', 'hour_minute', 'season',
        'substrate', 'substrate_details'
]

# Build a list of records, one per attribute
records = []

# 1) nfix_rate_converted -> attribute="rate_converted"
#    error from nfix_error_converted, error_type from nfix_error_type,
#    units from nfix_unit_converted
'''
records.append(nfix_df[id_cols].assign(
    attribute='rate_converted',
    value=nfix_df['nfix_rate_converted'],
    error=nfix_df['nfix_error_converted'],
    error_type=nfix_df['nfix_error_type'],
    units=nfix_df['nfix_unit_converted'],
))
'''

# 2) nfix_rate_original -> attribute="rate_original"
#    error from nfix_error_original, error_type from nfix_error_type,
#    units from nfix_unit_original
records.append(ground_truth_df[id_cols].assign(
    attribute='nfix_rate',
    value=ground_truth_df['nfix_rate_original'],
    error=ground_truth_df['nfix_error_original'],
    error_type=ground_truth_df['nfix_error_type'],
    units=ground_truth_df['nfix_unit_original'],
))

# 3) sample_depth -> attribute="sample_depth"
records.append(ground_truth_df[id_cols].assign(
    attribute='sample_depth',
    value=ground_truth_df['sample_depth'],
    error=np.nan,
    error_type=np.nan,
    units=np.nan,
))

# 4) nfix_method -> attribute="method"
records.append(ground_truth_df[id_cols].assign(
    attribute='nfix_method',
    value=ground_truth_df['nfix_method'],
    error=np.nan,
    error_type=np.nan,
    units=np.nan,
))

'''
# 5) nfix_incubation_time -> attribute="incubation_time"
records.append(nfix_df[id_cols].assign(
    attribute='incubation_time',
    value=nfix_df['nfix_incubation_time'],
    error=np.nan,
    error_type=np.nan,
    units=np.nan,
))
'''

# 6) nfix_incubation_temp -> attribute="incubation_temp"
records.append(ground_truth_df[id_cols].assign(
    attribute='nfix_incubation_temp_temperature',
    value=ground_truth_df['nfix_incubation_temp'],
    error=np.nan,
    error_type=np.nan,
    units=np.nan,
))

ground_truth_df = pd.concat(records, ignore_index=True)

# Reorder columns
ground_truth_df = ground_truth_df[id_cols + ['attribute', 'value', 'error', 'error_type', 'units']]
ground_truth_df = ground_truth_df.dropna(subset=['value'])

# Optionally sort so each original row's attributes are grouped together
ground_truth_df = ground_truth_df.sort_values(id_cols, kind='mergesort').reset_index(drop=True)

ground_truth_df = ground_truth_df.loc[ground_truth_df.attribute == 'nfix_rate']

In [3]:
ground_truth_df

,nfix_rate_id,reference_id,site_name,latitude,longitude,habitat,year,month,day,hour_minute,season,substrate,substrate_details,attribute,value,error,error_type,units
0,N1,R231,south west coast of Australia,-28.000000,114.000000,continental shelves,2003.0,10.0,NaN,NaN,NaN,wc,water,nfix_rate,0.017,0.006,stdev.,nmol-n l-1 h-1
3,N1027,R25,"East Weeks Bay, AL",30.400000,87.800000,estuaries,2012.0,2.0,NaN,NaN,NaN,benthos,sediment,nfix_rate,1.6,0.200,std. err,umol-n m-2 h-1
5,N1028,R25,"Weeks Bay Mouth, AL",30.400000,87.800000,estuaries,2012.0,2.0,NaN,NaN,NaN,benthos,sediment,nfix_rate,2.4,0.700,std. err,umol-n m-2 h-1
7,N1029,R25,"West Weeks Bay, AL",30.400000,87.800000,estuaries,2012.0,2.0,NaN,NaN,NaN,benthos,sediment,nfix_rate,2.6,0.400,std. err,umol-n m-2 h-1
9,N1030,R109,"Pearl River Estuary, South China Sea",22.747341,113.652219,estuaries,2018.0,7.0,31.0,NaN,NaN,benthos,sediment - inner estuary,nfix_rate,0.19,0.110,stdev.,ug-N g-1 d-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2919,N983,R173,"Lagunitas Creek, Tomales Bay, CA",38.171300,-122.912800,estuaries,1990.0,7.0,NaN,NaN,NaN,benthos,surface mudflat,nfix_rate,22.7,NaN,NaN,nmol-c2h4 cm-2 h-1
2921,N984,R173,"Lagunitas Creek, Tomales Bay, CA",38.171300,-122.912800,estuaries,1990.0,11.0,NaN,NaN,NaN,benthos,surface mudflat,nfix_rate,7.3,NaN,NaN,nmol-c2h4 cm-2 h-1
2923,N985,R173,"Lagunitas Creek, Tomales Bay, CA",38.171300,-122.912800,estuaries,1991.0,3.0,NaN,NaN,NaN,benthos,surface mudflat,nfix_rate,20.3,NaN,NaN,nmol-c2h4 cm-2 h-1
2925,N986,R173,"Lagunitas Creek, Tomales Bay, CA",38.171300,-122.912800,estuaries,1991.0,7.0,NaN,NaN,NaN,benthos,surface mudflat,nfix_rate,5.2,NaN,NaN,nmol-c2h4 cm-2 h-1


In [26]:
ground_truth_df.reference_id.value_counts().head(10)

reference_id
R163    121
R164     84
R172     42
R248     32
R124     31
R51      31
R59      26
R114     26
R43      25
R103     23
Name: count, dtype: int64

### Full, Extracted Dataset

In [6]:
# ---------------------------------
# Load experiment results
# ---------------------------------

experiment_data_path = "../data/experiments/2026_03_25/nfix_final.json"

unit_conversion_table = {
    'nfix_rate': {},
    'nfix_incubation_time': {},
    'nfix_incubation_temperature': {"°C": 1,},
}

attribute_types = {
    'nfix_rate': float,
    'nfix_incubation_time': float,
    'nfix_incubation_temperature': float,
}

# NOTE: some of these things you should get rid of in your extraction process!
drop_keys = ["feature_terms", "attribute_terms", "abbreviations", "table_logprob", "page_logprob", "judgement_raw_text"]
drop_attrs = ['nfix_incubation_time', 'nfix_incubation_temperature']

extracted_df = load_and_process_results(
    json_path=experiment_data_path,
    unit_conversion_table=unit_conversion_table,
    attribute_types=attribute_types,
    drop_keys=drop_keys,
    drop_attrs=drop_attrs,
    attribute_col="attribute",
    value_col="value",
    unit_col="units",
    out_col="processed_value"
)

# NOTE you need to change this to 'attribute'
extracted_df.rename(columns={"feature": "attribute"}, inplace=True)
extracted_df.sort_values(by=["title", "attribute"], inplace=True)
extracted_df.reset_index(drop=True, inplace=True)

#xtracted_df = extracted_df.loc[extracted_df.attribute == 'nfix_rate']
#extracted_df = extracted_df.reset_index(drop=True)

In [10]:
extracted_df.columns

Index(['reference', 'doi', 'doi_data', 'url', 'publication_year', 'authors',
       'title', 'publication', 'volume', 'pages', 'extraction_location',
       'extraction_location_details', 'paper_code', 'document_id', 'name',
       'location', 'latitude', 'longitude', 'habitat_type', 'substrate_type',
       'sample_depth', 'year', 'month', 'day', 'hour_minute', 'season',
       'entity_id', 'attribute', 'value', 'units', 'page_number',
       'table_number', 'row_index', 'column_index', 'source', 'context',
       'measurement_id', 'processed_value'],
      dtype='object')

In [12]:
extracted_df.sample_depth

0       surface
1          None
2          None
3          None
4          None
         ...   
1881       None
1882       None
1883       None
1884       None
1885       None
Name: sample_depth, Length: 1886, dtype: object

In [9]:
ground_truth_df

,nfix_rate_id,reference_id,site_name,latitude,longitude,habitat,year,month,day,hour_minute,season,substrate,substrate_details,attribute,value,error,error_type,units
0,N1,R231,south west coast of Australia,-28.000000,114.000000,continental shelves,2003.0,10.0,NaN,NaN,NaN,wc,water,nfix_rate,0.017,0.006,stdev.,nmol-n l-1 h-1
3,N1027,R25,"East Weeks Bay, AL",30.400000,87.800000,estuaries,2012.0,2.0,NaN,NaN,NaN,benthos,sediment,nfix_rate,1.6,0.200,std. err,umol-n m-2 h-1
5,N1028,R25,"Weeks Bay Mouth, AL",30.400000,87.800000,estuaries,2012.0,2.0,NaN,NaN,NaN,benthos,sediment,nfix_rate,2.4,0.700,std. err,umol-n m-2 h-1
7,N1029,R25,"West Weeks Bay, AL",30.400000,87.800000,estuaries,2012.0,2.0,NaN,NaN,NaN,benthos,sediment,nfix_rate,2.6,0.400,std. err,umol-n m-2 h-1
9,N1030,R109,"Pearl River Estuary, South China Sea",22.747341,113.652219,estuaries,2018.0,7.0,31.0,NaN,NaN,benthos,sediment - inner estuary,nfix_rate,0.19,0.110,stdev.,ug-N g-1 d-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2919,N983,R173,"Lagunitas Creek, Tomales Bay, CA",38.171300,-122.912800,estuaries,1990.0,7.0,NaN,NaN,NaN,benthos,surface mudflat,nfix_rate,22.7,NaN,NaN,nmol-c2h4 cm-2 h-1
2921,N984,R173,"Lagunitas Creek, Tomales Bay, CA",38.171300,-122.912800,estuaries,1990.0,11.0,NaN,NaN,NaN,benthos,surface mudflat,nfix_rate,7.3,NaN,NaN,nmol-c2h4 cm-2 h-1
2923,N985,R173,"Lagunitas Creek, Tomales Bay, CA",38.171300,-122.912800,estuaries,1991.0,3.0,NaN,NaN,NaN,benthos,surface mudflat,nfix_rate,20.3,NaN,NaN,nmol-c2h4 cm-2 h-1
2925,N986,R173,"Lagunitas Creek, Tomales Bay, CA",38.171300,-122.912800,estuaries,1991.0,7.0,NaN,NaN,NaN,benthos,surface mudflat,nfix_rate,5.2,NaN,NaN,nmol-c2h4 cm-2 h-1


### Match Extractions to Ground Truth

In [13]:
# Set of attributes which must be strictly equivalent to create a match
strict_matching = {
    "reference_id": "paper_code", # name in the ground truth dataset : name in the extracted dataset
    "attribute": "attribute",
    "value": "processed_value"
}

# Set of attributes which should be 
# compared by a fuzzy matching (roughly similar) to create a match.
fuzzy_matching = {
    "site_name": "name",
    #"substrate": "substrate_type",
    #"habitat": "habitat_type",
}

# This can take a while to run if you have a lot of data, 
# since it compares every extracted row to every ground truth row.
matching, matching_recall, matching_precision = matching_precision_recall(
    ground_truth_df,
    extracted_df,
    strict_matching=strict_matching,
    fuzzy_matching=fuzzy_matching,
    fuzzy_threshold = 0.0
)

print(f"Recall: {matching_recall:.4f}")
print(f"Precision: {matching_precision:.4f}")

Recall: 0.2452
Precision: 0.1288


### Debugging

In [18]:
gt_matched = np.array([False] * ground_truth_df.shape[0])
ex_matched = np.array([False] * extracted_df.shape[0])
for gt_idx, ex_idx in matching:
    gt_matched[gt_idx] = True
    ex_matched[ex_idx] = True

unmatched_gt = np.where(~gt_matched)[0]
unmatched_ex = np.where(~ex_matched)[0]

matched_gt_df = ground_truth_df[gt_matched == True]
unmatched_gt_df = ground_truth_df[gt_matched == False]
unmatched_gt_titles = unmatched_gt_df.title.value_counts().index

matched_ex_df = extracted_df[ex_matched == True]
unmatched_ex_df = extracted_df[ex_matched == False]
unmatched_ex_titles = unmatched_ex_df.title.value_counts().index

IndexError: index 6773 is out of bounds for axis 0 with size 3885

In [ ]:
title = "biodiversity of constructed wetlands for wastewater treatment"
gt_title_df = ground_truth_df.loc[ground_truth_df.title == title]
unmatched_gt_title_df = unmatched_gt_df.loc[unmatched_gt_df.title == title]
ex_title_df = extracted_df.loc[extracted_df.title == title]
unmatched_ex_title_df = unmatched_ex_df.loc[unmatched_ex_df.title == title]